In [1]:
import numpy as np
import random
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score


## Exercise 1: Hill-Climbing Search for Feature Selection in Predictive Modeling

Problem Statement: You are given a dataset (select a random dataset) with multiple input
features and a target variable. Each state represents a subset of selected features. The goal is
to find a feature subset that maximizes model accuracy.
State Representation

A binary vector indicating whether a feature is selected (1) or not (0)
Initial State

A randomly generated feature subset
Neighbor Generation

Flip one bit (add/remove one feature)
Evaluation Function

Cross-validation accuracy of a simple classifier (e.g., Logistic Regression)
Algorithm Implementation
Implement steepest-ascent hill climbing
Stop when no neighbor improves the current solution
Experimentation

Run the algorithm multiple times with different random initial states

In [25]:
class FeatureSelectionEnvironment:
    def __init__(self):
        data = load_breast_cancer()
        self.X = data.data
        self.y = data.target
        self.num_features = self.X.shape[1]

    def evaluate(self, state):
        if sum(state) == 0:
            return 0
        
        selected = [i for i in range(len(state)) if state[i] == 1]
        X_subset = self.X[:, selected]
        
        model = LogisticRegression(max_iter=5000)
        return cross_val_score(model, X_subset, self.y, cv=5).mean()


class HillClimbingAgent:
    def __init__(self, environment):
        self.env = environment
        # Random initial state
        self.state = np.random.randint(0, 2, self.env.num_features)

    def act(self):
        current_score = self.env.evaluate(self.state)

        while True:
            neighbors = []
import numpy as np
import random
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

            for i in range(len(self.state)):
                neighbor = self.state.copy()
                neighbor[i] = 1 - neighbor[i]
                neighbors.append(neighbor)

            scores = [self.env.evaluate(n) for n in neighbors]
            best_index = np.argmax(scores)

            if scores[best_index] > current_score:
                self.state = neighbors[best_index]
                current_score = scores[best_index]
            else:
                break

        return self.state, current_score


In [26]:
env = FeatureSelectionEnvironment()

num_runs = 5
best_overall_score = 0
best_overall_state = None

for run in range(num_runs):
    agent = HillClimbingAgent(env)   # New random initial state
    state, score = agent.act()

    print(f"Run {run+1}: Accuracy = {score:.4f}")

    if score > best_overall_score:
        best_overall_score = score
        best_overall_state = state

print("\nBest Overall Accuracy:", best_overall_score)


Run 1: Accuracy = 0.9561
Run 2: Accuracy = 0.9561
Run 3: Accuracy = 0.9543
Run 4: Accuracy = 0.9543
Run 5: Accuracy = 0.9543

Best Overall Accuracy: 0.9560627231796305


## Exercise 2: Simulated Annealing for Vehicle Routing Optimization

Problem Statement: A delivery vehicle must visit a set of locations and return to the depot.
The order of visiting locations affects the total travel cost. Each state represents a permutation
of delivery locations.
State Representation:
A sequence representing the visiting order of delivery points

Initial State:
A random permutation of locations

Neighbor Generation:
Swap two locations in the route

Cost Function:
Total travel distance of the route
Simulated Annealing Implementation
Implement SA using:
oTemperature schedule T(t)
oAcceptance probability:
−Δ E/ T

In [23]:
import math

class VehicleRoutingEnvironment:
    def __init__(self, num_cities=10):
        self.num_cities = num_cities
        self.cities = np.random.rand(num_cities, 2) * 100

    def route_cost(self, route):
        total = 0
        for i in range(len(route)):
            total += np.linalg.norm(
                self.cities[route[i]] -
                self.cities[route[(i+1) % len(route)]]
            )
        return total


class SimulatedAnnealingAgent:
    def __init__(self, environment):
        self.env = environment
        self.state = list(range(self.env.num_cities))
        random.shuffle(self.state)

    def act(self):
        T = 1000
        cooling_rate = 0.995
        
        current_cost = self.env.route_cost(self.state)
        
        while T > 1:
            neighbor = self.state.copy()
            i, j = random.sample(range(len(neighbor)), 2)
            neighbor[i], neighbor[j] = neighbor[j], neighbor[i]
            
            neighbor_cost = self.env.route_cost(neighbor)
            delta = neighbor_cost - current_cost
            
            if delta < 0 or random.random() < math.exp(-delta / T):
                self.state = neighbor
                current_cost = neighbor_cost
            
            T *= cooling_rate
        
        return self.state, current_cost


# Run Agent
env = VehicleRoutingEnvironment()
agent = SimulatedAnnealingAgent(env)

route, cost = agent.act()
print("Final Cost:", cost)



Final Cost: 284.3981243272902


## Exercise 3 (Revised): Genetic Algorithm for Autonomous Drone Path Planning

Problem Statement: A drone must travel from a start location to a destination in a 2D
environment with obstacles. Each candidate solution represents a sequence of waypoints
defining a flight path. The goal is to evolve paths that minimize travel distance while
avoiding obstacles.
Chromosome Representation

A fixed-length sequence of (x, y) waypoints
Initial Population

Randomly generated feasible paths
Fitness Function

Minimize:
oTotal path length
oObstacle collisions (high penalty)
Genetic Operators
Selection: Tournament selection
Crossover: One-point crossover on waypoint sequence
Mutation: Random perturbation of waypoint coordinates
Constraints
Waypoints must remain within map boundaries
Paths intersecting obstacles are penalized
Termination Condition

Fixed number of generations or convergence

In [24]:
class DroneEnvironment:
    def __init__(self):
        self.map_size = 100
        self.start = np.array([0, 0])
        self.end = np.array([100, 100])
        self.obstacle_center = np.array([50, 50])
        self.obstacle_radius = 15

    def fitness(self, path):
        total = 0
        full = [self.start] + path + [self.end]
        
        for i in range(len(full)-1):
            total += np.linalg.norm(full[i] - full[i+1])
        
        penalty = 0
        for point in path:
            if np.linalg.norm(point - self.obstacle_center) < self.obstacle_radius:
                penalty += 1000
        
        return total + penalty


class GeneticDroneAgent:
    def __init__(self, environment, pop_size=30, waypoints=5):
        self.env = environment
        self.pop_size = pop_size
        self.waypoints = waypoints
        self.population = [
            [np.random.rand(2)*100 for _ in range(waypoints)]
            for _ in range(pop_size)
        ]

    def tournament(self):
        candidates = random.sample(self.population, 3)
        candidates.sort(key=lambda x: self.env.fitness(x))
        return candidates[0]

    def evolve(self, generations=100):
        for _ in range(generations):
            new_population = []
            
            for _ in range(self.pop_size):
                p1 = self.tournament()
                p2 = self.tournament()
                
                point = random.randint(1, self.waypoints-1)
                child = p1[:point] + p2[point:]
                
                for i in range(self.waypoints):
                    if random.random() < 0.1:
                        child[i] = np.clip(
                            child[i] + np.random.randn(2)*5,
                            0, 100
                        )
                
                new_population.append(child)
            
            self.population = new_population
        
        best = min(self.population, key=lambda x: self.env.fitness(x))
        return best, self.env.fitness(best)


# Run Agent
env = DroneEnvironment()
agent = GeneticDroneAgent(env)

best_path, cost = agent.evolve()
print("Best Path Cost:", cost)


Best Path Cost: 141.4552043607673
